# Module 322 — Guardrails Challenge (SnackOverflow)

This notebook mirrors the HTB challenge contract locally. It contains no target address, credentials, or submission call. Set `SUBMIT = True` only after you have an active authorized HTB target and have reviewed the exact request format.

## Requirements translated into code
1. **Input:** remove disallowed characters; keep the allowed URL punctuation; truncate to 512 characters; raise `GuardrailException` if the prompt contains `packetsnacc.local`.
2. **Output:** parse JSON; require `type` and `response`; permit only `text` and `url`; for URLs permit only `http` or `https`; HTML-escape text.

The code is a sequence of predicates and transformations rather than a model attack. A predicate is a yes/no test. A transformation maps one string to another, for example `prompt[:512]` maps a long prompt to its first 512 characters.

In [ ]:
import html
import json
import re
from urllib.parse import urlparse

# HTB supplies this exception in the challenge runner. The fallback lets us test locally.
class GuardrailException(Exception):
    pass

MAX_PROMPT_LENGTH = 512
COMPETITOR_DOMAIN = "packetsnacc.local"


## Input guardrail

Read `[^A-Za-z0-9\s.:/\-_@]` as: **a character that is not** a letter, digit, whitespace, or one of `. : / - _ @`. `re.sub` removes each matching character by replacing it with the empty string.

The order matters. First reject the competitor domain, then sanitize and truncate. Otherwise a clever punctuation change could alter what is detected before policy enforcement. The domain check is case-insensitive because hostnames are case-insensitive.

In [ ]:
def input_guardrail(prompt: str) -> str:
    """Apply the challenge's input policy and return the safe prompt."""
    if not isinstance(prompt, str):
        raise GuardrailException("Invalid prompt type")

    # Policy predicate: competitor requests are blocked, not repaired.
    if COMPETITOR_DOMAIN in prompt.casefold():
        raise GuardrailException("Invalid URL")

    # Formula-like rule: output = sanitize(input), where each disallowed
    # character contributes the empty string. This silently removes symbols.
    cleaned = re.sub(r"[^A-Za-z0-9\s.:/\-_@]", "", prompt)

    # Length constraint: keep indices 0 through 511 (at most 512 chars).
    return cleaned[:MAX_PROMPT_LENGTH]


## Output guardrail

`json.loads` converts the wire-format string into a Python value. We then validate the object like a small schema. For a URL response, `urlparse` separates the scheme from the rest; accepting only `http` and `https` prevents local-file schemes such as `file:///etc/passwd`. For text, `html.escape(..., quote=True)` converts markup delimiters and quotes into safe entities, preventing them from being interpreted as HTML.

In [ ]:
def _valid_http_url(value: str) -> bool:
    parsed = urlparse(value)
    # A URL must have an allowed scheme and a network location/host.
    return parsed.scheme.casefold() in {"http", "https"} and bool(parsed.netloc)


def output_guardrail(response: str) -> str:
    """Validate and, for text, safely encode the model response."""
    try:
        value = json.loads(response)
    except (TypeError, json.JSONDecodeError) as exc:
        raise GuardrailException("Invalid JSON") from exc

    if not isinstance(value, dict) or not {"type", "response"}.issubset(value):
        raise GuardrailException("Invalid JSON")

    kind = value["type"]
    body = value["response"]
    if kind not in {"text", "url"} or not isinstance(body, str):
        raise GuardrailException("Invalid JSON")

    if kind == "url":
        if not _valid_http_url(body):
            if urlparse(body).scheme.casefold() not in {"http", "https"}:
                raise GuardrailException("Invalid URL scheme")
            raise GuardrailException("Invalid URL")
    else:
        # Preserve the schema but replace dangerous HTML characters.
        value["response"] = html.escape(body, quote=True)

    # Compact JSON is accepted by the challenge and is easy to compare.
    return json.dumps(value, separators=(",", ":"))


In [ ]:
# Deterministic local tests; no network and no HTB submission.
assert input_guardrail("Hello World!") == "Hello World"
assert input_guardrail("Hello, please summarize https://academy.hackthebox.com/") == "Hello please summarize https://academy.hackthebox.com/"
assert len(input_guardrail("A" * 600)) == 512
try:
    input_guardrail("visit https://packetsnacc.local/")
except GuardrailException:
    pass
else:
    raise AssertionError("competitor domain was not blocked")

assert output_guardrail('{"type": "text", "response": "Test<>\""}') == '{"type":"text","response":"Test&lt;&gt;&quot;"}'
assert output_guardrail('{"type": "url", "response": "https://academy.hackthebox.com/"}') == '{"type":"url","response":"https://academy.hackthebox.com/"}'
for bad in ["Test", "{}", '{"type": "invalid", "response": ""}', '{"type": "url", "response": "file:///etc/passwd"}']:
    try:
        output_guardrail(bad)
    except GuardrailException:
        pass
    else:
        raise AssertionError(f"accepted invalid response: {bad}")

print("All local guardrail tests passed.")


In [ ]:
# Submission remains intentionally disabled until explicit authorization.
SUBMIT = False
TARGET = "INSTANCE_IP:PORT"  # fill this only for an active authorized HTB target
if SUBMIT:
    raise NotImplementedError("Review the live HTB request contract before enabling submission.")
else:
    print("Submission disabled; notebook is running offline tests only.")
